# Modelo MILP de Keccak Adaptativo basado en intentos fallidos

## 1. Introducción

Keccak es el algoritmo de cifrado utilizado como base del estándar SHA-3. Su estructura interna está formada por una función de permutación aplicada sobre un estado de 5×5 lanes, donde cada lane contiene z bits.

En este trabajo se propone una variante adaptativa de Keccak donde el número de intentos fallidos de autenticación modifica dinámicamente la seguridad del algoritmo.

La idea principal es incrementar el costo computacional cuando se detectan intentos repetidos de acceso, aumentando:

- el número de rondas de Keccak.
- la constante utilizada en la operación Iota.
- la dimensión efectiva del estado mediante un hipercubo dinámico.

La variable dinámica se define como:

$$
\alpha = \frac{2^f}{\sqrt{d}}
$$

donde:

- $f$: número de intentos fallidos.
- $d$: número de dimensiones del estado.

El número adaptativo de rondas se define como:

$$
R=\min(24,8+\lfloor \alpha \rfloor)
$$

Esto permite mantener pocas rondas para usuarios legítimos y aumentar la seguridad frente a ataques automatizados.

Sobre el criptoanalisis
> **Nota sobre versiones anteriores:**  
> En el primer intento (intentos/MILP_AES_Keccak_gurobi.ipynb) se busco traducir solo el código utilizado para la librería gurobi, sin embargo, excedía el número de variables, se paso al 2do intento.
> En la versión del programa del 2do intento (intentos/MILP_AES_Keccak_gurobi_2.ipynb) se incluyó, además de la optimización MILP,
> una estimación de probabilidad diferencial basada únicamente en:
>
> $$
> P=(0.5)^n
> $$
>
> donde $n$ representaba el número de S-boxes activas.
>
> Esta aproximación no representa correctamente la probabilidad diferencial de la
> capa Chi de Keccak, debido a que dicha probabilidad depende de las transiciones
> diferenciales específicas de la S-box. Por ello, esta parte fue retirada y el
> análisis MILP_AES_Keccak_gurobi_3.ipynb se centró únicamente en la métrica MILP de minimización de S-boxes activas.

# 2. Keccak Adaptativo Propuesto
Cuaderno SHA3adaptativoSLRO.ipynb
La variante propuesta modifica principalmente la etapa Iota:

Keccak original:

$$
A_{0,0}=A_{0,0}\oplus RC_i
$$


Keccak adaptativo:

$$
A_{0,0}=A_{0,0}\oplus(RC_i\oplus\lfloor\alpha\rfloor)
$$


De esta manera los intentos fallidos generan una perturbación adicional en las constantes de ronda.

Además se utiliza un salt dinámico para evitar ataques basados en tablas arcoíris.

Proceso:

1. Generación de salt mediante timestamp.
2. Concatenación:

$$
mensaje=salt || password
$$

3. Aplicación de Keccak adaptativo.
4. Almacenamiento:

$$
(salt,hash,intentos\_fallidos)
$$


# 3. Modelo MILP de Keccak Reducido

Para analizar la resistencia diferencial se construyó un modelo MILP de una versión reducida de Keccak.

Configuración utilizada:

| Parámetro | Valor |
|-|-|
| Estado | 5×5 lanes |
| Tamaño de lane | z = 4 bits |
| Estado total | 100 bits |
| Rondas evaluadas | 1,2 |
| Solver | Gurobi |
| Variable objetivo | mínimo número de S-boxes activas |


Cada bit del estado se representa mediante una variable binaria:

$$
S_{r,x,y,z}\in\{0,1\}
$$


donde:

- $r$: ronda.
- $x,y$: posición dentro del estado.
- $z$: posición del bit.

# 4. Modelamiento MILP de las capas de Keccak reducido

El modelo implementado representa una versión reducida de Keccak-f mediante
Programación Lineal Entera Mixta (MILP), con el objetivo de encontrar el número
mínimo de S-boxes activas en la capa Chi.

Para reducir la complejidad computacional se utiliza un tamaño de palabra:

$$
z=4
$$

por lo que el estado tiene:

$$
5 \times 5 \times 4 = 100
$$

bits, correspondiente a un Keccak reducido de 100 bits.

Cada bit del estado se representa mediante una variable binaria:

$$
S^{r}_{x,y,z}\in\{0,1\}
$$

donde:

- $r$: índice de ronda.
- $x,y$: coordenadas del lane dentro del estado.
- $z$: posición del bit dentro del lane.

---

# 4.1 Capa Theta

La transformación Theta calcula primero la paridad de cada columna del estado:

$$
C[x,z]=
\bigoplus_{y=0}^{4}S[x,y,z]
$$


En el modelo MILP, la operación XOR se transforma mediante restricciones
lineales binarias.

Para dos variables binarias:

$$
out=a\oplus b
$$

se utilizan las restricciones:

$$
out\leq a+b
$$

$$
out\geq a-b
$$

$$
out\geq b-a
$$

$$
out\leq 2-a-b
$$


Posteriormente se calcula:

$$
D[x,z]
=
C[x-1,z]
\oplus
C[x+1,z-1]
$$


Finalmente, el estado después de Theta es:

$$
B[x,y,z]
=
S[x,y,z]
\oplus
D[x,z]
$$


---

# 4.2 Capa Rho-Pi

Las transformaciones Rho y Pi se modelan mediante restricciones de igualdad,
ya que únicamente realizan permutaciones y rotaciones de bits.

La rotación de cada lane utiliza los desplazamientos definidos por la tabla
de rotaciones de Keccak:

$$
\rho[x,y]
$$


La redistribución de posiciones se representa como:

$$
B[x',y',z']
=
A[x,y,z]
$$


donde:

- $(x,y)$ representa la posición original del lane.
- $(x',y')$ representa la nueva posición después de Pi.
- $z'$ representa el desplazamiento producido por Rho.

En el modelo MILP esta transformación solamente agrega igualdades entre
variables binarias.

---

# 4.3 Capa Chi

La capa Chi es la única transformación no lineal de Keccak.

La operación original se define como:

$$
A'_i=
A_i
\oplus
((\neg A_{i+1})\land A_{i+2})
$$


Para representar la operación AND mediante MILP se introduce una variable
auxiliar:

$$
T_i=
(\neg A_{i+1})\land A_{i+2}
$$


La variable $T_i$ se modela mediante:

$$
T_i\leq A_{i+2}
$$


$$
T_i\leq 1-A_{i+1}
$$


$$
T_i\geq A_{i+2}-A_{i+1}
$$


Finalmente:

$$
A'_i=A_i\oplus T_i
$$


La operación XOR se implementa mediante las restricciones lineales binarias
definidas anteriormente.

---

# 4.4 Detección de S-boxes activas

Cada operación Chi trabaja sobre cinco bits de una columna.

Una S-box Chi se considera activa cuando al menos uno de sus cinco bits de
entrada presenta actividad diferencial.

Se define:

$$
ChiActive[y,z]
$$


tal que:

$$
ChiActive[y,z]
=
OR_x(ChiInput[x,y,z])
$$


La condición se modela mediante:

$$
ChiActive[y,z]\geq ChiInput[x,y,z]
$$


y:

$$
ChiActive[y,z]
\leq
\sum_x ChiInput[x,y,z]
$$


Por lo tanto:

- Si todos los bits son cero:

$$
ChiActive=0
$$

- Si al menos un bit es activo:

$$
ChiActive=1
$$


---

# 4.5 Función objetivo

El objetivo del modelo MILP es minimizar la cantidad total de S-boxes Chi
activas durante las rondas consideradas:

$$
\min
\sum_{r,y,z}ChiActive[r,y,z]
$$


El valor obtenido por Gurobi representa el mínimo número de S-boxes activas
para la trayectoria diferencial encontrada.

---

# 4.6 Configuración experimental

Para reducir la complejidad computacional se utiliza:

| Parámetro | Valor |
|---|---|
| Número de lanes | $5\times5$ |
| Bits por lane | $z=4$ |
| Tamaño del estado | 100 bits |
| Solucionador | Gurobi MILP |
| Objetivo | Minimizar S-boxes activas |

El modelo permite estudiar la difusión de Keccak reducido y observar cómo
aumenta el número mínimo de S-boxes activas al incrementar el número de rondas.

# 5. Experimento MILP sobre Keccak reducido

El objetivo del experimento es validar el modelo MILP desarrollado para una
versión reducida de Keccak y obtener el número mínimo de S-boxes Chi activas
para una ronda de transformación.

Debido al crecimiento del número de variables binarias y restricciones del
modelo MILP, el experimento se realizó inicialmente considerando una sola ronda
de Keccak reducido.

Experimento 2 de 2 Rondas, no soportó el número de variables, por parte de la licencia académica.

---

# 5.1 Configuración experimental

El modelo fue evaluado con los siguientes parámetros:

| Parámetro | Valor |
|---|---|
| Algoritmo | Keccak reducido |
| Número de rondas | 1 |
| Organización del estado | $5\times5$ lanes |
| Bits por lane | $z=4$ |
| Tamaño del estado | 100 bits |
| Variables del modelo | Binarias |
| Modelo de optimización | MILP |
| Solucionador | Gurobi |
| Tiempo límite | 300 segundos |
| Función objetivo | Minimizar S-boxes Chi activas |

---

# 5.2 Metodología experimental

El experimento se realizó siguiendo los siguientes pasos:

1. Se definieron las variables binarias que representan el estado inicial y
   final de Keccak:

$$
S^0_{x,y,z}
$$

y

$$
S^1_{x,y,z}
$$


2. Se estableció que la diferencia de entrada y salida fueran no nulas:

$$
\sum_{x,y,z}S^0_{x,y,z}\geq1
$$


$$
\sum_{x,y,z}S^1_{x,y,z}\geq1
$$


3. Se aplicaron las restricciones MILP correspondientes a las capas:

- Theta.
- Rho-Pi.
- Chi.

4. Se definió la función objetivo para minimizar las S-boxes activas:

$$
Min
\sum_{y,z}ChiActive[y,z]
$$


5. Finalmente, el modelo fue resuelto utilizando Gurobi y se obtuvo el número
   mínimo de S-boxes activas para una ronda.

---

# 5.3 Resultado experimental

El modelo MILP fue ejecutado utilizando una ronda de Keccak reducido con un
tamaño de estado de 100 bits, considerando lanes organizados como una matriz
$5\times5$ y un tamaño de palabra de $z=4$ bits.

La configuración utilizada fue:

| Parámetro | Valor |
|---|---|
| Rondas | 1 |
| Lanes | $5\times5$ |
| Bits por lane | $z=4$ |
| Tamaño del estado | 100 bits |
| Variables binarias iniciales | 560 |
| Restricciones MILP | 1842 |
| Solucionador | Gurobi 13.0.2 |
| Tiempo límite | 300 segundos |

Durante la optimización, Gurobi redujo el modelo mediante la etapa de presolve,
obteniendo un problema equivalente con 1202 restricciones y 380 variables
binarias.

El modelo encontró una solución óptima:

$$
ObjVal=1
$$

Por lo tanto, el número mínimo de S-boxes Chi activas para una ronda es:

$$
\boxed{N_{S-box}=1}
$$


El resultado indica que para una sola ronda de Keccak reducido existe una
trayectoria diferencial con actividad mínima en una única S-box Chi.

El solucionador confirmó la optimalidad de la solución con:

$$
Gap=0\%
$$

y un tiempo de ejecución de:

$$
t=0.389\ segundos
$$


La distribución de actividad obtenida fue:

| Ronda | S-boxes Chi activas |
|---|---|
| 0 | 1 |

Estos resultados validan la correcta implementación del modelo MILP para una
ronda reducida de Keccak. Sin embargo, una sola ronda representa únicamente una
validación inicial del modelo, ya que la propiedad de difusión de Keccak se
manifiesta al analizar un mayor número de rondas.


======================================================================
MILP KECCAK REDUCIDO - GUROBI
Resultado de celda
======================================================================
Rondas: 1
Lanes: 5x5
Bits por lane: 4
Estado: 100 bits

Creando variables...
Variables creadas:
560

Agregando restricciones iniciales...
Procesando ronda 0

Restricciones completadas

Resolviendo...
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 1842 rows, 640 columns and 5260 nonzeros (Min)
Model fingerprint: 0x34f74209
Model has 20 linear objective coefficients
Variable types: 0 continuous, 640 integer (640 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve removed 640 rows and 260 columns
Presolve time: 0.04s
Presolved: 1202 rows, 380 columns, 4120 nonzeros
Variable types: 0 continuous, 380 integer (380 binary)
Found heuristic solution: objective 19.0000000
Found heuristic solution: objective 2.0000000

Root relaxation: objective 1.000000e-01, 694 iterations, 0.02 seconds (0.03 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    0.10000    0  223    2.00000    0.10000  95.0%     -    0s
     0     0    0.33333    0  238    2.00000    0.33333  83.3%     -    0s
H    0     0                       1.0000000    1.00000  0.00%     -    0s
     0     0    1.00000    0  268    1.00000    1.00000  0.00%     -    0s

Cutting planes:
  Gomory: 1
  Clique: 39
  MIR: 7
  Zero half: 1
  Mod-K: 1
  RLT: 19
  BQP: 2

Explored 1 nodes (1677 simplex iterations) in 0.37 seconds (0.18 work units)
Thread count was 8 (of 8 available processors)

Solution count 3: 1 2 19 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.000000000000e+00, best bound 1.000000000000e+00, gap 0.0000%

======================================================================
RESULTADOS
======================================================================
Estado: OPTIMAL
SolCount: 3
ObjVal: 1.0

MINIMO DE S-BOXES ACTIVAS: 1.0

Por ronda:
Ronda 0: 1

Variables activas en estado inicial:
S[0,0,0,0] = 1
S[0,0,1,0] = 1
S[0,0,2,0] = 1
S[0,0,2,3] = 1
S[0,0,3,0] = 1
S[0,0,4,0] = 1
S[0,1,0,1] = 1
S[0,1,0,3] = 1
S[0,1,1,1] = 1
S[0,1,1,3] = 1
S[0,1,2,0] = 1
S[0,1,2,1] = 1
S[0,1,2,3] = 1
S[0,1,3,1] = 1
S[0,1,3,3] = 1
S[0,1,4,1] = 1
S[0,1,4,3] = 1
S[0,2,0,0] = 1
S[0,2,1,0] = 1
S[0,2,2,0] = 1
S[0,2,2,3] = 1
S[0,2,3,0] = 1
S[0,2,4,0] = 1
S[0,3,0,0] = 1
S[0,3,0,2] = 1
S[0,3,1,0] = 1
S[0,3,1,2] = 1
S[0,3,2,0] = 1
S[0,3,2,2] = 1
S[0,3,3,0] = 1
S[0,3,3,2] = 1
S[0,3,4,0] = 1
S[0,3,4,2] = 1
S[0,4,0,1] = 1
S[0,4,0,2] = 1
S[0,4,1,1] = 1
S[0,4,1,2] = 1
S[0,4,2,1] = 1
S[0,4,2,2] = 1
S[0,4,3,1] = 1
S[0,4,3,2] = 1
S[0,4,4,1] = 1
S[0,4,4,2] = 1

Tiempo: 0.38927268981933594 segundos
======================================================================


